# The Panel: Logs, Notes, and Tables

Each bar has a panel for text that belongs with the progress:

- `bar.log(...)` appends a message tagged with its level, like `logging`
- `bar.write(...)` prints plain text permanently above the bars, like `print()`
- `bar.panel(content)` replaces the note shown with the bars

`panel()` also lays out structured data such as dicts, dataclasses, and DataFrames as a table. Run the cells to watch the tables update live.

In [1]:
import io
import logging
import time
from collections import namedtuple
from dataclasses import dataclass

import numpy as np
import pandas as pd
import polars as pl

import psnek

pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_hide_column_data_types(True)
rng = np.random.default_rng(7)

## Text: Log, Panel, and Write

This bar sets `Options(panel_clear=True)`, so its transient status is removed once the loop ends.

In [2]:
with psnek.Bar(total=40, desc="crawl", options=psnek.Options(panel_clear=True)) as bar:
    for page in range(40):
        time.sleep(0.04)
        if page % 10 == 0:
            bar.log(f"fetched sitemap page {page}")
        bar.panel(f"current page: {page}\nqueue depth: {40 - page}")
        bar.update()
psnek.write("crawl finished: 40 pages")

info  fetched sitemap page 0
  info  fetched sitemap page 10
  info  fetched sitemap page 20
  info  fetched sitemap page 30
# crawl #################################  100.0%      40/40         0:01   22.9it/s  done

crawl finished: 40 pages


## Log Levels and Timestamps

`log()` is for messages about the run, like the `logging` module: each line starts with its level (`debug`, `info`, `warn`, `error`, or `fatal`, colored like the bar states in a terminal), `Options(log_time=True)` puts the time first, and the lines of a longer message are indented under the first. Records from a `logging` logger get the same tags through `psnek.LogHandler`.

`write()` prints its text as-is, like `print()`, so it suits results such as a saved checkpoint. Logged lines wait in the panel below the bars until newer lines push them above, while written lines go above the bars at once. In a notebook, all of it stays in the cell's one output, alongside the bars.

In [3]:
logger = logging.getLogger("ingest")
logger.propagate = False
logger.handlers = [psnek.LogHandler(log_time=True)]

options = psnek.Options(log_time=True)
with psnek.Bar(total=30, desc="ingest", options=options) as bar:
    for batch in range(30):
        time.sleep(0.03)
        if batch == 4:
            bar.log("schema inferred: 12 columns", level=logging.DEBUG)
        elif batch == 10:
            bar.log("batch 10 ingested")
        elif batch == 15:
            bar.write("checkpoint saved: ingest-015.parquet")
        elif batch == 18:
            bar.log("batch 18: 3 malformed rows skipped", level=logging.WARNING)
        elif batch == 24:
            logger.error("upload of batch 24 failed\nretrying in 1s")
        bar.update()

09:31:40 debug schema inferred: 12 columns
  09:31:40 info  batch 10 ingested
checkpoint saved: ingest-015.parquet
# ingest ################################  100.0%      30/30         0:01   29.6it/s  done
  09:31:41 warn  batch 18: 3 malformed rows skipped
  09:31:41 error upload of batch 24 failed
                 retrying in 1s

## A Dict: Live Metrics

Keys are bold and aligned, and floats are shown to four significant digits. Calling `panel()` with the same content as before does nothing, so it's cheap to call every step.

In [4]:
with psnek.Bar(total=80, desc="fine-tune") as bar:
    for step in range(1, 81):
        time.sleep(0.03)
        bar.panel(
            {
                "step": step,
                "loss": 2.4 / step**0.5,
                "learning_rate": 3e-4 * min(1.0, step / 20),
                "grad_norm": None if step < 10 else 1.0 + 0.5 * np.sin(step / 7),
                "best_checkpoint": f"checkpoints/step-{step // 20 * 20:04d}.pt",
            }
        )
        bar.update()

# fine-tune #############################  100.0%      80/80         0:02   29.2it/s  done
| step             80
| loss             0.2683
| learning_rate    0.0003000
| grad_norm        0.5461
| best_checkpoint  checkpoints/step-0080.pt

## A List of Dicts: A Table

The keys become the header, and a row missing a key leaves its cell blank. `Options(panel_justify="right")` aligns every column to the right, which suits numeric tables.

In [5]:
workers = [
    {"worker": f"gpu-{index}", "state": "busy", "batches": 0, "last_error": None}
    for index in range(4)
]
options = psnek.Options(panel_justify="right")
with psnek.Bar(total=200, desc="distributed eval", options=options) as bar:
    for batch in range(200):
        time.sleep(0.01)
        worker = workers[batch % 4]
        worker["batches"] += 1
        worker["state"] = "busy" if batch < 190 else "draining"
        if batch == 120:
            workers[3]["last_error"] = "CUDA OOM, retried"
        bar.panel(workers)
        bar.update()

# distributed eval ######################  100.0%    200/200         0:02   80.0it/s  done
| worker     state  batches         last_error
|  gpu-0  draining       50               None
|  gpu-1  draining       50               None
|  gpu-2  draining       50               None
|  gpu-3  draining       50  CUDA OOM, retried

## Dataclasses and Named Tuples

Dataclasses and named tuples work too, either alone or in a list.

In [6]:
@dataclass
class Shard:
    """An index shard and its build progress."""

    name: str
    documents: int
    size_mb: float


Throughput = namedtuple("Throughput", "docs_per_s mb_per_s")
shards = [Shard(f"shard-{index:02d}", 0, 0.0) for index in range(5)]
with psnek.Bar(total=150, desc="build index") as bar:
    for step in range(150):
        time.sleep(0.015)
        shard = shards[step % 5]
        shard.documents += 1000
        shard.size_mb += 3.7
        bar.panel(shards if step < 120 else Throughput(66_000, 245.3))
        bar.update()

# build index ###########################  100.0%    150/150         0:02   55.8it/s  done
| docs_per_s  66000
| mb_per_s    245.3

## Long Tables

Only the rows that fit are formatted, even for a list of a million rows, and a last line counts the rest. Long cells are cut to 40 characters.

In [7]:
files = [
    {"path": f"data/raw/2026/10/part-{index:07d}.parquet", "rows": index * 17}
    for index in range(1_000_000)
]
options = psnek.Options(panel_lines=5)
with psnek.Bar(total=60, desc="scan files", options=options) as bar:
    for step in range(60):
        time.sleep(0.02)
        bar.panel(files[step * 1000 :])
        bar.update()

# scan files ############################  100.0%      60/60         0:01   30.4it/s  done
| path                                   rows
| data/raw/2026/10/part-0059000.parquet  1003000
| data/raw/2026/10/part-0059001.parquet  1003017
| data/raw/2026/10/part-0059002.parquet  1003034
| ... 940997 more rows

## polars and pandas DataFrames

DataFrames are shown as tables too, although psnek never imports pandas or polars itself. Here, a simulated hyperparameter sweep keeps a live leaderboard as a polars DataFrame.

In [8]:
trials: list[dict[str, float | int | str]] = []
with psnek.Bar(total=40, desc="sweep (polars)") as bar:
    for trial in range(40):
        time.sleep(0.05)
        learning_rate = float(10 ** rng.uniform(-5, -2))
        trials.append(
            {
                "trial": trial,
                "optimizer": rng.choice(["adamw", "sgd", "lion"]).item(),
                "learning_rate": learning_rate,
                "val_loss": float(
                    abs(np.log10(learning_rate) + 3.5) * 0.2 + rng.normal(0.3, 0.02)
                ),
            }
        )
        bar.panel(pl.DataFrame(trials).sort("val_loss").head(5))
        bar.update()

leaderboard = pl.DataFrame(trials).sort("val_loss")
print(leaderboard.head(5))

# sweep (polars) ########################  100.0%      40/40         0:02   18.0it/s  done
| trial  optimizer  learning_rate  val_loss
| 19     adamw      0.0003337      0.2918
| 10     lion       0.0003505      0.2982
| 12     sgd        0.0003095      0.3048
| 6      sgd        0.0004576      0.3229
| 5      adamw      0.0002164      0.3324

┌───────┬───────────┬───────────────┬──────────┐
│ trial ┆ optimizer ┆ learning_rate ┆ val_loss │
╞═══════╪═══════════╪═══════════════╪══════════╡
│ 19    ┆ adamw     ┆ 0.000334      ┆ 0.291834 │
│ 10    ┆ lion      ┆ 0.00035       ┆ 0.298159 │
│ 12    ┆ sgd       ┆ 0.000309      ┆ 0.304812 │
│ 6     ┆ sgd       ┆ 0.000458      ┆ 0.322946 │
│ 5     ┆ adamw     ┆ 0.000216      ┆ 0.332369 │
└───────┴───────────┴───────────────┴──────────┘


A pandas DataFrame with a meaningful index shows the index as its leading columns.

In [9]:
classes = ["cat", "dog", "bird", "fish"]
correct = pd.Series(0, index=pd.Index(classes, name="label"))
seen = pd.Series(0, index=correct.index)
with psnek.Bar(total=120, desc="evaluate (pandas)") as bar:
    for _ in range(120):
        time.sleep(0.02)
        labels = rng.choice(classes, size=32)
        hits = rng.random(32) < np.where(labels == "fish", 0.7, 0.92)
        seen = seen.add(pd.Series(labels).value_counts(), fill_value=0)
        correct = correct.add(pd.Series(labels[hits]).value_counts(), fill_value=0)
        report = pd.DataFrame({"seen": seen.astype(int), "accuracy": correct / seen})
        bar.panel(report)
        bar.update()

print(report)

# evaluate (pandas) #####################  100.0%    120/120         0:03   39.0it/s  done
| label  seen  accuracy
| bird   965   0.9244
| cat    979   0.9234
| dog    930   0.9215
| fish   966   0.6791

       seen  accuracy
label                
bird    965  0.924352
cat     979  0.923391
dog     930  0.921505
fish    966  0.679089


## Tables in Plain Logs

In log mode, such as in CI, the panel's content is printed once after it changes. Writing the bars to a `StringIO` shows that output here, with each `panel_justify` setting.

In [10]:
top = leaderboard.select("trial", "optimizer", "learning_rate", "val_loss").head(3)
for justify in ("left", "right", "center"):
    log = io.StringIO()
    options = psnek.Options(stream=log, panel_justify=justify)
    with psnek.Bar(total=3, desc=f"justify={justify}", options=options) as bar:
        bar.panel(top)
        bar.update(3)
    print(log.getvalue())

justify=left: 0.0% 0/3 0:00<--:-- --it/s
justify=left: 100.0% 3/3 0:00 done 3916it/s
trial  optimizer  learning_rate  val_loss
19     adamw      0.0003337      0.2918
10     lion       0.0003505      0.2982
12     sgd        0.0003095      0.3048



justify=right: 0.0% 0/3 0:00<--:-- --it/s
justify=right: 100.0% 3/3 0:00 done 4077it/s
trial  optimizer  learning_rate  val_loss
   19      adamw      0.0003337    0.2918
   10       lion      0.0003505    0.2982
   12        sgd      0.0003095    0.3048



justify=center: 0.0% 0/3 0:00<--:-- --it/s
justify=center: 100.0% 3/3 0:00 done 5541it/s
trial  optimizer  learning_rate  val_loss
 19      adamw      0.0003337     0.2918
 10      lion       0.0003505     0.2982
 12       sgd       0.0003095     0.3048

